In [ ]:
import os

# D diskinin içinde 'hf_cache' diye bir klasör oluşturup her şeyi oraya yönlendiriyoruz.
# Eğer diskinin adı E ise, "E:/hf_cache" olarak değiştir.
os.environ["HF_HOME"] = "D:/hf_cache" 

print("✅ Hugging Face indirme dizini değiştirildi:", os.environ["HF_HOME"])

In [ ]:
from datasets import load_dataset, Dataset

print("⏳ Sentetik veri seti indiriliyor...")
# Hızlı eğitim ve deneme için ilk 3000 veriyi alıyoruz. İstersen artırabilirsin.
dataset = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes", split="train[:3000]")

training_data = []
for item in dataset:
    # Modelin belleğini taşırmamak için not uzunluğunu sınırlandırıyoruz
    note = item['note'][:3000] 
    
    # Modelin doktor rolüne girmesi için instruction şablonu
    instruction = f"""You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Patient ID: {item.get('patient_id', 'Unknown')}
- Task: Discharge Summary generation based on clinical context.

Write a structured clinical note with visit details, diagnosis, treatment, and discharge plan."""

    # LLaMA / Meditron formatına uygun birleştirme
    formatted_text = f"### Instruction:\n{instruction}\n\n### Response:\n{note}"
    
    training_data.append({"text": formatted_text})

hf_dataset = Dataset.from_list(training_data)

print(f"✅ {len(hf_dataset)} sentetik eğitim örneği hazırlandı!")

In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets trl huggingface_hub

In [ ]:
import os
from huggingface_hub import login

# Token: ortam degiskeni HF_TOKEN veya huggingface-cli login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace'e giriş yapıldı!")
else:
    print("⚠️ HF_TOKEN tanımlı değil; public modeller için giriş atlanabilir.")

In [ ]:
from datasets import load_dataset, Dataset

print("⏳ Sentetik veri seti indiriliyor...")
# Hızlı eğitim ve deneme için ilk 3000 veriyi alıyoruz. İstersen artırabilirsin.
dataset = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes", split="train[:3000]")

training_data = []
for item in dataset:
    # Modelin belleğini taşırmamak için not uzunluğunu sınırlandırıyoruz
    note = item['note'][:3000] 
    
    # Modelin doktor rolüne girmesi için instruction şablonu
    instruction = f"""You are a medical doctor. Write a detailed clinical discharge note (epikriz) for the following patient.

Patient Information:
- Patient ID: {item.get('patient_id', 'Unknown')}
- Task: Discharge Summary generation based on clinical context.

Write a structured clinical note with visit details, diagnosis, treatment, and discharge plan."""

    # LLaMA / Meditron formatına uygun birleştirme
    formatted_text = f"### Instruction:\n{instruction}\n\n### Response:\n{note}"
    
    training_data.append({"text": formatted_text})

hf_dataset = Dataset.from_list(training_data)

print(f"✅ {len(hf_dataset)} sentetik eğitim örneği hazırlandı!")

In [ ]:
!pip install ipywidgets hf_xet

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Modeli VRAM dostu, 3 Milyar parametreli Qwen ile değiştirdik!
model_name = "Qwen/Qwen2.5-3B-Instruct"

# Önce ekran kartını görüyor mu diye test edelim
print(f"Ekran kartı devrede mi?: {torch.cuda.is_available()}")

# 4-bit Quantization ayarları
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # fp16 daha güvenlidir
)

print("⏳ Qwen-3B indiriliyor ve yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto" 
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model hazır! Eğitilecek parametre oranı: %{100*trainable/total:.2f}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Modeli VRAM dostu, 3 Milyar parametreli Qwen ile değiştirdik!
model_name = "Qwen/Qwen2.5-3B-Instruct"

# Heyecanlı an: Ekran kartını görüyor mu?
is_cuda_active = torch.cuda.is_available()
print(f"Ekran kartı devrede mi?: {is_cuda_active}")

if is_cuda_active:
    # 4-bit Quantization ayarları
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16 # fp16 daha güvenlidir
    )

    print("⏳ Qwen-3B indiriliyor ve yükleniyor...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token 

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto" 
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"✅ Model hazır! Eğitilecek parametre oranı: %{100*trainable/total:.2f}")
    print("🚀 Artık SFTTrainer (Hücre 5) ile eğitime başlayabilirsin!")
else:
    print("❌ HATA: CUDA hala aktif değil. Lütfen üst menüden Kernel -> Restart yaptığından emin ol.")

In [ ]:
import torch
import os
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

print("⏳ Sistem hazırlanıyor, 4-bit sıkıştırma protokolü devrede...")

# 1. HAYAT KURTARAN AYAR: Modeli test ederken de 4 GB'a sığdırmak için sıkıştırıyoruz
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Çelik kasadaki 15 saatlik yedeği buluyoruz
ana_klasor = "./qwen-epikriz-model"
yedekler = [os.path.join(ana_klasor, d) for d in os.listdir(ana_klasor) if d.startswith("checkpoint")]
en_son_yedek = sorted(yedekler, key=lambda x: int(x.split("-")[-1]))[-1]
print(f"📥 15 Saatlik Emek Bulundu! Yüklenen Kasa: {en_son_yedek}")

# 3. Sözlüğü ve Modeli SIKIŞTIRILMIŞ (4-bit) halde çağırıyoruz
tokenizer = AutoTokenizer.from_pretrained(en_son_yedek)
model = AutoPeftModelForCausalLM.from_pretrained(
    en_son_yedek,
    device_map="auto",
    quantization_config=bnb_config  # Kernel'in ölmesini engelleyen kalkanımız!
)

model.config.use_cache = True
model.eval()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

print("✅ Model RAM'i patlatmadan başarıyla yüklendi! Rapor yazılıyor...\n")

# 4. Nadir Hastalık (Wilson) Testi
hasta_sikayeti = "Hasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu."

prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel ve tıbbi terminolojiye uygun bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs,
        max_new_tokens=1024,      
        temperature=0.3,          
        repetition_penalty=1.2,   
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)

print("-" * 40)
print("📝 UZMAN DOKTOR QWEN'İN EPİKRİZ RAPORU:")
print("-" * 40)
print(sonuc_metni.split("assistant\n")[-1])

In [ ]:
import torch
import os
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("1. Aşama: Konfigürasyonlar okunuyor...")
ana_klasor = "./qwen-epikriz-model"
yedekler = [os.path.join(ana_klasor, d) for d in os.listdir(ana_klasor) if d.startswith("checkpoint")]
en_son_yedek = sorted(yedekler, key=lambda x: int(x.split("-")[-1]))[-1]

# Eğittiğimiz klasörün içinden temel modelin (Qwen) adını çekiyoruz
with open(os.path.join(en_son_yedek, "adapter_config.json"), "r") as f:
    peft_config = json.load(f)
base_model_name = peft_config["base_model_name_or_path"]

print(f"2. Aşama: Temel Model ({base_model_name}) 4-bit olarak belleğe alınıyor...")
# Sadece temel modeli 4-bit sıkıştırmayla alıyoruz (RAM'i yormaz, sistem çökmez)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(en_son_yedek)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("3. Aşama: 15 Saatlik Emek (LoRA Ağırlıkları) modele enjekte ediliyor...")
# Şimdi senin eğittiğin doktorluk yeteneklerini temel modelin üstüne giydiriyoruz!
model = PeftModel.from_pretrained(base_model, en_son_yedek)

model.config.use_cache = True
model.eval()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

print("✅ Başarı! Darboğaz aşıldı, epikriz raporu üretiliyor...\n")

# --- TEST KISMI ---
hasta_sikayeti = "Hasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu."

prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel ve tıbbi terminolojiye uygun bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs,
        max_new_tokens=1024,      
        temperature=0.3,          
        repetition_penalty=1.2,   
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)

print("-" * 40)
print("📝 UZMAN DOKTOR QWEN'İN EPİKRİZ RAPORU:")
print("-" * 40)
print(sonuc_metni.split("assistant\n")[-1])

In [ ]:
import torch
import os
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer

print("1. Veri seti ve Ayarlar yükleniyor...")
dataset = load_dataset("json", data_files="nadir_hastaliklar.json", split="train")

ana_klasor = "./qwen-epikriz-model"
yedekler = [os.path.join(ana_klasor, d) for d in os.listdir(ana_klasor) if d.startswith("checkpoint")]
en_son_yedek = sorted(yedekler, key=lambda x: int(x.split("-")[-1]))[-1]

with open(os.path.join(en_son_yedek, "adapter_config.json"), "r") as f:
    base_model_name = json.load(f)["base_model_name_or_path"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(en_son_yedek)
tokenizer.pad_token = tokenizer.eos_token

print("2. Temel Model RAM'i koruyarak çağrılıyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("3. 15 Saatlik 'Pratisyen' Hekim belleğe alınıyor ve EĞİTİME AÇILIYOR...")
# is_trainable=True ile önceki emeğimizin üstüne yeni bilgileri (JSON) yazabileceğiz!
model = PeftModel.from_pretrained(base_model, en_son_yedek, is_trainable=True)
model = prepare_model_for_kbit_training(model)

egitim_ayarlari = TrainingArguments(
    output_dir="./qwen-uzman-doktor",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=15, # Sadece 2 hastamız olduğu için 15 kez okuyup iyice kavrasın
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=egitim_ayarlari
)

print("🚀 UZMANLIK EĞİTİMİ BAŞLIYOR! (Maksimum 1-2 dakika sürecek)...")
trainer.train()

print("✅ Eğitim bitti! Şimdi GERÇEK bir Uzman Doktor test ediliyor...\n")

model.config.use_cache = True
model.eval()

hasta_sikayeti = "Hasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu."
prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel ve tıbbi terminolojiye uygun bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs,
        max_new_tokens=512,      
        temperature=0.1,          # Yaratıcılığı iyice kıstık ki bir daha uydurmasın!
        repetition_penalty=1.2,   
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)

print("-" * 40)
print("🎓 GERÇEK UZMAN DOKTOR QWEN'İN EPİKRİZ RAPORU:")
print("-" * 40)
print(sonuc_metni.split("assistant\n")[-1])

In [ ]:
import json

# Uzmanlık Veri Setini Yeniden Oluşturuyoruz
ozel_veri_seti = [
    {
        "text": "<|im_start|>user\nHasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu.<|im_end|>\n<|im_start|>assistant\nEPİKRİZ RAPORU\n\nHASTA BİLGİLERİ:\nYaş/Cinsiyet: 22 / Erkek\nBaşvuru Şikayeti: İstirahat tremoru, dizartri, disfaji.\n\nKLİNİK VE LABORATUVAR BULGULARI:\nNörolojik muayenede ekstrapiramidal bulgular saptandı. Oftalmolojik muayenede bilateral Kayser-Fleischer halkası pozitifliği doğrulandı. Biyokimyasal incelemede; serum seruloplazmin düzeyi düşük, 24 saatlik idrar bakır atılımı ise belirgin yüksek saptanmıştır.\n\nTANI VE SÜREÇ:\nMevcut klinik, oftalmolojik ve biyokimyasal bulgular ışığında hastaya 'Wilson Hastalığı (Hepatolentiküler Dejenerasyon)' tanısı konulmuştur. Hasta nöroloji ve gastroenteroloji bölümleri eşliğinde multidisipliner takibe alınmıştır. Bakır şelasyon tedavisine (D-penisilamin veya Trientin) başlanması planlanmaktadır.<|im_end|>"
    },
    {
        "text": "<|im_start|>user\nHasta 45 yaşında kadın. Son 6 aydır artan proksimal kas güçsüzlüğü, özellikle merdiven çıkarken zorlanma ve yutma güçlüğü şikayeti var. Fizik muayenede heliotrop döküntü (göz kapaklarında morumsu kızarıklık) ve el parmak eklemleri üzerinde Gottron papülleri mevcut. Kreatin kinaz (CK) seviyesi 4000 U/L.<|im_end|>\n<|im_start|>assistant\nEPİKRİZ RAPORU\n\nHASTA BİLGİLERİ:\nYaş/Cinsiyet: 45 / Kadın\nBaşvuru Şikayeti: Proksimal kas güçsüzlüğü, disfaji, ciltte döküntü.\n\nKLİNİK VE LABORATUVAR BULGULARI:\nNörolojik muayenede simetrik proksimal kas güçsüzlüğü (4/5) izlendi. Dermatolojik incelemede periorbital heliotrop döküntü ve Gottron papülleri saptandı. Laboratuvar tetkiklerinde kas enzimleri (CK: 4000 U/L) belirgin yüksek bulunmuştur.\n\nTANI VE SÜREÇ:\nKlinik prezentasyon ve laboratuvar bulguları 'Dermatomiyozit' ile uyumludur. Tanının kesinleştirilmesi amacıyla kas biyopsisi, EMG ve miyozit spesifik antikor (MSA) paneli istenmiştir. Akut inflamasyonu baskılamak için yüksek doz sistemik kortikosteroid tedavisi başlanmış olup, hasta romatoloji kliniğine yatırılmıştır.<|im_end|>"
    }
]

with open("nadir_hastaliklar.json", "w", encoding="utf-8") as f:
    json.dump(ozel_veri_seti, f, ensure_ascii=False, indent=4)

print("✅ Eksik dosya başarıyla D diskine oluşturuldu! Artık eğitime geçebiliriz.")

In [ ]:
import torch
import os
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig 

print("1. Veri seti ve Ayarlar yükleniyor...")
dataset = load_dataset("json", data_files="nadir_hastaliklar.json", split="train")

ana_klasor = "./qwen-epikriz-model"
yedekler = [os.path.join(ana_klasor, d) for d in os.listdir(ana_klasor) if d.startswith("checkpoint")]
en_son_yedek = sorted(yedekler, key=lambda x: int(x.split("-")[-1]))[-1]

with open(os.path.join(en_son_yedek, "adapter_config.json"), "r") as f:
    base_model_name = json.load(f)["base_model_name_or_path"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(en_son_yedek)
tokenizer.pad_token = tokenizer.eos_token

print("2. Temel Model RAM'i koruyarak çağrılıyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("3. 15 Saatlik 'Pratisyen' Hekim belleğe alınıyor ve EĞİTİME AÇILIYOR...")
model = PeftModel.from_pretrained(base_model, en_son_yedek, is_trainable=True)
model = prepare_model_for_kbit_training(model)

# O baş belası max_seq_length parametresini tamamen sildik!
egitim_ayarlari = SFTConfig(
    output_dir="./qwen-uzman-doktor",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=15, 
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none",
    dataset_text_field="text" 
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=egitim_ayarlari 
)

print("🚀 UZMANLIK EĞİTİMİ BAŞLIYOR! (Maksimum 1-2 dakika sürecek)...")
trainer.train()

print("✅ Eğitim bitti! Şimdi GERÇEK bir Uzman Doktor test ediliyor...\n")

model.config.use_cache = True
model.eval()

hasta_sikayeti = "Hasta 22 yaşında erkek. Ellerde istirahat tremoru, konuşmada bozulma (dizartri) ve yutma güçlüğü şikayetleriyle başvurdu. Göz muayenesinde korneada Kayser-Fleischer halkası izlendi. Laboratuvar testlerinde serum seruloplazmin seviyesi çok düşük, 24 saatlik idrar bakır atılımı belirgin şekilde artmış bulundu."
prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel ve tıbbi terminolojiye uygun bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs,
        max_new_tokens=512,      
        temperature=0.1,          
        repetition_penalty=1.2,   
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)

print("-" * 40)
print("🎓 GERÇEK UZMAN DOKTOR QWEN'İN EPİKRİZ RAPORU:")
print("-" * 40)
print(sonuc_metni.split("assistant\n")[-1])

In [ ]:
import json
import random

# --- TIBBİ TERİM HAVUZLARI ---
# Bu havuzları ne kadar doldurursak, model o kadar zeki olur.
hastaliklar = [
    {"tani": "Wilson Hastalığı", "bulgu": "Kayser-Fleischer halkası, düşük seruloplazmin, tremor, dizartri", "bolum": "Gastroenteroloji / Nöroloji"},
    {"tani": "Dermatomiyozit", "bulgu": "Heliotrop döküntü, Gottron papülleri, proksimal kas güçsüzlüğü, yüksek CK", "bolum": "Romatoloji"},
    {"tani": "Sistemik Lupus Eritematozus (SLE)", "bulgu": "Malar rash (kelebek döküntü), eklem ağrısı, ANA pozitifliği, proteinüri", "bolum": "Romatoloji"},
    {"tani": "Akromegali", "bulgu": "Büyümüş el ve ayaklar, kaba yüz hatları, yüksek büyüme hormonu (GH), IGF-1 yüksekliği", "bolum": "Endokrinoloji"},
    {"tani": "Huntington Hastalığı", "bulgu": "Koreiform hareketler, kognitif yıkım, kaudat çekirdek atrofisi", "bolum": "Nöroloji"}
]

cinsiyetler = ["Erkek", "Kadın"]
yaslar = range(18, 85)

def veri_uret(adet=500):
    yeni_veri = []
    for _ in range(adet):
        h = random.choice(hastaliklar)
        yas = random.choice(yaslar)
        cins = random.choice(cinsiyetler)
        
        # Kullanıcı (Doktor) Sorusu Şablonu
        user_text = f"Hasta {yas} yaşında {cins.lower()}. Şikayetleri: {h['bulgu']}. Laboratuvar ve fizik muayene sonuçlarına göre profesyonel bir epikriz yazar mısın?"
        
        # Asistan (Uzman Doktor) Cevap Şablonu - İŞTE QWEN BURAYI EZBERLEYECEK
        assistant_text = f"""EPİKRİZ RAPORU

HASTA BİLGİLERİ:
Yaş/Cinsiyet: {yas} / {cins}
Ön Tanı: {h['tani']}

KLİNİK BULGULAR:
Hastanın yapılan fizik muayenesinde {h['bulgu']} saptanmıştır. Klinik tablo ilgili uzmanlık birimi olan {h['bolum']} tarafından değerlendirilmiştir.

TANI VE PLAN:
Mevcut semptomatoloji ve laboratuvar parametreleri doğrultusunda '{h['tani']}' tanısı üzerinde durulmaktadır. Hastanın multidisipliner takibi planlanmış olup, ileri tetkik ve tedavi protokolü başlatılmıştır."""

        full_text = f"<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n{assistant_text}<|im_end|>"
        yeni_veri.append({"text": full_text})
    
    return yeni_veri

# 1000 adet kusursuz tıbbi veri üretiyoruz
mega_dataset = veri_uret(1000)

with open("nadir_hastaliklar.json", "w", encoding="utf-8") as f:
    json.dump(mega_dataset, f, ensure_ascii=False, indent=4)

print(f"✅ Fabrika çalıştı! 1000 satırlık dev tıbbi veri seti 'nadir_hastaliklar.json' hazır.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_yolu = r"D:\epikriz_model\41d298d8-3345-4b3e-b570-583920fa4b2f\qwen-uzman-doktor-final"
base_model_adi = "Qwen/Qwen2.5-7B-Instruct" 
cache_klasoru = r"D:\huggingface_cache"

print("⏳ Uzman Doktor hafızaya alınıyor (İndirme bitti, direkt diskten okunuyor)...")

# ÇÖZÜM BURADA: CPU'ya taşma (offload) izni verdik!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True  # EKLENEN SİHİRLİ SATIR
)

tokenizer = AutoTokenizer.from_pretrained(model_yolu, cache_dir=cache_klasoru)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_adi,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    cache_dir=cache_klasoru
)

model = PeftModel.from_pretrained(base_model, model_yolu)
model.eval()

print("✅ Model başarıyla yüklendi. Artık tıbbi raporlar için hazırız!")

# --- TEST KISMI ---
def epikriz_yaz(sikayet):
    prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel bir epikriz raporu yaz: {sikayet}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
    
    return tokenizer.decode(output[0], skip_special_tokens=True).split("assistant\n")[-1]

yeni_hasta = "Hasta 30 yaşında kadın, kelebek döküntüsü ve eklem ağrısı ile başvurdu."
print("\n📝 TEST RAPORU:\n", epikriz_yaz(yeni_hasta))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_yolu = r"D:\epikriz_model\41d298d8-3345-4b3e-b570-583920fa4b2f\qwen-uzman-doktor-final"

# DİKKAT: Artık doğrudan D diskindeki temel modeli de okuması için yolu tam veriyoruz.
# Eğer o 15GB'lık temel Qwen modeli D diskinde 'qwen-uzman-doktor-final' klasörünün içinde DEĞİLSE,
# o indirdiğin klasörün yolunu buraya yazmalısın. Eğer emin değilsen bunu bu şekilde çalıştır:
base_model_adi = "Qwen/Qwen2.5-7B-Instruct" 

print("⏳ Uzman Doktor hafızaya alınıyor (İnternet bağlantısı KESİLDİ, SADECE DİSK)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True 
)

# ÇÖZÜM: local_files_only=True ekledik. Artık internetten BİR BAYT BİLE İNDİREMEZ.
tokenizer = AutoTokenizer.from_pretrained(
    model_yolu, 
    local_files_only=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_adi,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    local_files_only=True # İNTERNET YASAK!
)

model = PeftModel.from_pretrained(base_model, model_yolu)
model.eval()

print("✅ Model başarıyla yüklendi. Artık tıbbi raporlar için hazırız!")

# --- TEST KISMI ---
def epikriz_yaz(sikayet):
    prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel bir epikriz raporu yaz: {sikayet}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
    
    return tokenizer.decode(output[0], skip_special_tokens=True).split("assistant\n")[-1]

yeni_hasta = "Hasta 30 yaşında kadın, kelebek döküntüsü ve eklem ağrısı ile başvurdu."
print("\n📝 TEST RAPORU:\n", epikriz_yaz(yeni_hasta))

In [1]:
import shutil
import os

# Hedef tahtasındaki (silinecek) devasa klasörler
silinecek_hedefler = [
    # 1. D Diskine indirdiğimiz o devasa 15 GB'lık model
    r"D:\huggingface_cache\models--Qwen--Qwen2.5-7B-Instruct",
    
    # 2. Hata veren o eski uzman doktor katmanı
    r"D:\epikriz_model\41d298d8-3345-4b3e-b570-583920fa4b2f\qwen-uzman-doktor-final",
    
    # 3. C diskini boğan o ilk yarım kalmış indirmeler (Gizli .cache klasörü)
    os.path.expanduser(r"~\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct")
]

print("🧹 D-CAD Temizlik Protokolü Başlatılıyor...\n")

Kazanilan_Alan_Tahmini = 15 # GB cinsinden tahmini

for yol in silinecek_hedefler:
    if os.path.exists(yol):
        try:
            # Klasörü içindeki her şeyle birlikte (ağaç yapısı) siler
            shutil.rmtree(yol)
            print(f"✅ BAŞARIYLA SİLİNDİ: {yol}")
        except PermissionError:
            print(f"⚠️ HATA: Dosya silinemedi ({yol}). Jupyter'de 'Kernel -> Restart' yaptığından emin ol!")
        except Exception as e:
            print(f"❌ BEKLENMEYEN HATA: {yol} -> {e}")
    else:
        print(f"⏭️ BULUNAMADI (Zaten temiz): {yol}")

print(f"\n🎉 Temizlik tamamlandı! Bilgisayarın az önce yaklaşık {Kazanilan_Alan_Tahmini} GB'lık bir yükten kurtuldu.")

🧹 D-CAD Temizlik Protokolü Başlatılıyor...

✅ BAŞARIYLA SİLİNDİ: D:\huggingface_cache\models--Qwen--Qwen2.5-7B-Instruct
✅ BAŞARIYLA SİLİNDİ: D:\epikriz_model\41d298d8-3345-4b3e-b570-583920fa4b2f\qwen-uzman-doktor-final
✅ BAŞARIYLA SİLİNDİ: C:\Users\ASUS\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct

🎉 Temizlik tamamlandı! Bilgisayarın az önce yaklaşık 15 GB'lık bir yükten kurtuldu.


In [1]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# 1. HAFİF VE HIZLI MODELİ SEÇİYORUZ (Sadece ~3 GB inecek, RAM'i yormayacak)
base_model_adi = "Qwen/Qwen2.5-1.5B-Instruct" 
cache_klasoru = r"D:\huggingface_cache"

print("1. Veri seti yükleniyor (1000 satırlık Nadir Hastalıklar)...")
# Dosyanın aynı klasörde olduğundan emin ol
dataset = load_dataset("json", data_files="nadir_hastaliklar.json", split="train")

print("2. Mini Qwen (1.5B) indiriliyor/yükleniyor (Çok hızlı olacak)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_adi, cache_dir=cache_klasoru)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_adi,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=cache_klasoru
)

print("3. Model eğitime hazırlanıyor...")
base_model = prepare_model_for_kbit_training(base_model)

# Yeni (hafif) bedene göre yeni LoRA ayarı
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)

egitim_ayarlari = SFTConfig(
    output_dir="./qwen-mini-uzman",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=3, # 1000 veriyi 3 kez okuyacak
    logging_steps=10,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=egitim_ayarlari
)

print("🚀 HIZLI UZMANLIK EĞİTİMİ BAŞLIYOR! (Çayını al, arkana yaslan)...")
trainer.train()

# Eğitilen Mini Doktoru D diskine güvenle kaydedelim
kayit_yolu = r"D:\epikriz_model\qwen-mini-uzman-final"
trainer.model.save_pretrained(kayit_yolu)
tokenizer.save_pretrained(kayit_yolu)
print(f"\n✅ Eğitim Bitti! Mini Uzman Doktor '{kayit_yolu}' adresine mühürlendi.")

# --- MEYVAYI TOPLAMA VAKTİ: TEST KISMI ---
print("\n🔍 DOKTOR UYANDI, TEST EDİLİYOR...")
model.config.use_cache = True
model.eval()

hasta_sikayeti = "Hasta 30 yaşında kadın, kelebek döküntüsü, eklem ağrısı ve halsizlik şikayetiyle başvurdu. Kan tahlilinde ANA pozitif."
prompt = f"<|im_start|>user\nŞu hasta şikayetine göre profesyonel bir epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(**inputs, max_new_tokens=512, temperature=0.1)

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=True)
print("\n📝 TEST RAPORU:\n", sonuc_metni.split("assistant\n")[-1])

1. Veri seti yükleniyor (1000 satırlık Nadir Hastalıklar)...
2. Mini Qwen (1.5B) indiriliyor/yükleniyor (Çok hızlı olacak)...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

3. Model eğitime hazırlanıyor...


...

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 HIZLI UZMANLIK EĞİTİMİ BAŞLIYOR! (Çayını al, arkana yaslan)...


Step,Training Loss
10,2.944076
20,2.197234
30,1.521661
40,0.895694
50,0.610294
60,0.343751
70,0.307714
80,0.227535
90,0.131558
100,0.133398



✅ Eğitim Bitti! Mini Uzman Doktor 'D:\epikriz_model\qwen-mini-uzman-final' adresine mühürlendi.

🔍 DOKTOR UYANDI, TEST EDİLİYOR...

📝 TEST RAPORU:
 HASTA EPİKRİZ RAPORU

Hasta: 
Yaş/Cinsiyet: 30 / Kadın
Öze: Kelebek Döküntü, Eklem Ağrısı ve Halsizlig Rachip.requireNonNull
Kan Tahlili: ANA Pozitif

Rapodunuzla zarf edilecek baki etkiler: Helterisel kan hazirleri, mazmuran siniri ve meyar sinevi. Profesyonel ve akidetic bir epikriz raporunu yazınız.


In [2]:
import torch

# Doktorun uydurmasını engelleyen ve şablonu zorlayan katı test:
model.eval()

hasta_sikayeti = "Hasta 30 yaşında kadın, kelebek döküntüsü, eklem ağrısı ve halsizlik şikayetiyle başvurdu. Kan tahlilinde ANA pozitif."

# Sistemi zorlamak için System prompt ekledik ve cevaba "EPİKRİZ RAPORU" diye zorla biz başlattık.
prompt = f"<|im_start|>system\nSen uzman bir Türk doktorsun. Asla uydurma kelimeler veya kod yazmazsın. Sadece net ve kısa epikriz raporu yazarsın.<|im_end|>\n<|im_start|>user\nŞu hasta şikayetine göre epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\nEPİKRİZ RAPORU\n\nHASTA BİLGİLERİ:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs, 
        max_new_tokens=150,       # Çok uzun yazıp saçmalamasını engelledik
        temperature=0.01,         # Yaratıcılığı SIFIRA indirdik (Uydurması yasak)
        repetition_penalty=1.2,   # Aynı şeyi tekrarlamasını engelledik
        eos_token_id=tokenizer.im_end_id # <|im_end|> görünce SUS!
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=False)
temiz_sonuc = sonuc_metni.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "")

print("\n🚨 SIKI YÖNETİM TEST RAPORU:\n", temiz_sonuc)

AttributeError: Qwen2Tokenizer has no attribute im_end_id

In [3]:
import torch

# Doktorun uydurmasını engelleyen ve şablonu zorlayan katı test:
model.eval()

hasta_sikayeti = "Hasta 30 yaşında kadın, kelebek döküntüsü, eklem ağrısı ve halsizlik şikayetiyle başvurdu. Kan tahlilinde ANA pozitif."

prompt = f"<|im_start|>system\nSen uzman bir Türk doktorsun. Asla uydurma kelimeler veya kod yazmazsın. Sadece net ve kısa epikriz raporu yazarsın.<|im_end|>\n<|im_start|>user\nŞu hasta şikayetine göre epikriz raporu yaz: {hasta_sikayeti}<|im_end|>\n<|im_start|>assistant\nEPİKRİZ RAPORU\n\nHASTA BİLGİLERİ:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# HATA DÜZELTİLDİ: Sözlükten <|im_end|> token'ının gerçek ID'sini çekiyoruz
durma_kodu = tokenizer.convert_tokens_to_ids("<|im_end|>")

with torch.no_grad():
    cikti = model.generate(
        **inputs, 
        max_new_tokens=150,       # Çok uzun yazıp saçmalamasını engelledik
        temperature=0.01,         # Yaratıcılığı SIFIRA indirdik (Uydurması yasak)
        repetition_penalty=1.2,   # Aynı şeyi tekrarlamasını engelledik
        eos_token_id=durma_kodu   # <|im_end|> görünce SUS!
    )

sonuc_metni = tokenizer.decode(cikti[0], skip_special_tokens=False)
temiz_sonuc = sonuc_metni.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "")

print("\n🚨 SIKI YÖNETİM TEST RAPORU:\n", temiz_sonuc)


🚨 SIKI YÖNETİM TEST RAPORU:
 EPİKRİZ RAPORU

HASTA BİLGİLERİ:
Üzeman: Dr. Xxx (Türk Doktor)
Mevcut Tazar: Epicrisis

Rapodan After:
- Hastada Koreselin Koremorfleri üzerine bir epikrizrapo sundu.
- Uzman firma cekevecek mi?
- Mevcut Takas: Dr. YYY (Türk Doctordan)

Tasarım Ayarları:
Stili: Klassik - Şekilli
Vid: Instagram - TikTok
Lang: English - German


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Temiz, eğitilmemiş, orijinal zeki modeli çağırıyoruz (Saniyeler içinde açılır)
model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
cache_klasoru = r"D:\huggingface_cache"

print("⏳ Temiz ve Orijinal Qwen (1.5B) hafızaya alınıyor...")

tokenizer = AutoTokenizer.from_pretrained(model_adi, cache_dir=cache_klasoru)
model = AutoModelForCausalLM.from_pretrained(
    model_adi,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=cache_klasoru
)
model.eval()
print("✅ Qwen uyandı! Şimdi ona bir ajan (agent) gibi davranacağız.")

# --- FEW-SHOT PROMPTING (Örnekli Yönlendirme) ---
# Modele eğitim vermek yerine, sistem mesajıyla ona bir kimlik ve kesin bir şablon veriyoruz.

hasta_sikayeti = "Hasta 30 yaşında kadın, kelebek döküntüsü, eklem ağrısı ve halsizlik şikayetiyle başvurdu. Kan tahlilinde ANA pozitif."

prompt = f"""<|im_start|>system
Sen alanında uzman, son derece ciddi ve resmi bir Türk Tıp Doktorusun. 
Görevin, verilen hasta şikayetlerini analiz edip KESİNLİKLE aşağıdaki şablona uyarak epikriz raporu yazmaktır.
Şablonun dışına çıkmak, yorum yapmak, kod yazmak veya sosyal medya (Instagram vb.) kelimeleri kullanmak YASAKTIR.

ŞABLON:
EPİKRİZ RAPORU
-----------------
HASTA BİLGİLERİ:
- Yaş/Cinsiyet: [Yaş ve Cinsiyet]
- Başvuru Şikayeti: [Şikayetlerin özeti]

KLİNİK VE LABORATUVAR BULGULARI:
- [Bulgular]

TANI VE PLAN:
- [Tıbbi Ön Tanı ve yapılacaklar]
<|im_end|>
<|im_start|>user
Şu hasta verisine göre şablonu doldur: {hasta_sikayeti}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    cikti = model.generate(
        **inputs, 
        max_new_tokens=200,
        temperature=0.1, # Yaratıcılık kapalı, sadece şablona sadık kalacak
        top_p=0.9
    )

sonuc = tokenizer.decode(cikti[0], skip_special_tokens=True)
temiz_rapor = sonuc.split("assistant\n")[-1]

print("\n🩺 GERÇEK DOKTORUN RAPORU:\n", temiz_rapor)

⏳ Temiz ve Orijinal Qwen (1.5B) hafızaya alınıyor...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


✅ Qwen uyandı! Şimdi ona bir ajan (agent) gibi davranacağız.

🩺 GERÇEK DOKTORUN RAPORU:
 EPİKRİZ RAPORU
-----------------
HASTA BİLGİLERİ:
- Yaş/Cinsiyet: 30 yaş kadın
- Başvuru Şikayeti: Kelebek döküntüsü, eklem ağrısı ve halsizlik

KLİNİK VE LABORATUVAR BULGULARI:
- Kan tahlili: ANA pozitif

TANI VE PLAN:
- Kelebek döküntüsü: Dikkatli inceleniyor
- Eklem ağrısı: İstiraz edici, daha fazla bilgi gerekiyor
- Halsizlik: Hastanın durumunu kontrol etmek için laboratuvar çalışması gerekiyor


In [6]:
import torch

# Modelimiz zaten hafızada, sadece yeni ve çok daha profesyonel bir prompt veriyoruz.
model.eval()

# Asistanın (yani bizim veya sistemin) topladığı veriler. 
# Küçük modellere tanıyı veya terimleri hafifçe fısıldarsak, muazzam bir edebi rapor yazarlar.
asistan_notu = "Hasta 30 yaşında kadın. Yüzünde güneşe çıkınca artan kelebek tarzı makülopapüler döküntü mevcut. El bileklerinde ve dizlerinde şişlik ve eklem ağrısı (poliartralji) var. Son 2 aydır şiddetli yorgunluk tarifliyor. Laboratuvar: ANA (Antinükleer Antikor) pozitif, Anti-dsDNA pozitif geldi. Hastada Sistemik Lupus Eritematozus (SLE) düşünüyoruz. Uygun tedavi başlandı."

# Orijinal modeli "Kıdemli Uzman Doktor" rolüne sokan devasa şablonumuz:
prompt = f"""<|im_start|>system
Sen Türkiye'nin en iyi eğitim ve araştırma hastanelerinden birinde görev yapan, 20 yıllık tecrübeye sahip bir Romatoloji Uzmanısın.
Görevin, asistan doktorun sana verdiği kısa notları kullanarak, akademik tıbbi terminolojiye tam uygun, detaylı, gerçekçi ve profesyonel bir "KLİNİK EPİKRİZ RAPORU" yazmaktır.
Asla uydurma kelime (örneğin Rachip, İstiraz vb.) kullanma. Aşağıdaki detaylı şablonu eksiksiz doldur ve her bölümü uzun uzun açıkla.

ŞABLON:
=========================================
T.C. SAĞLIK BAKANLIĞI
HASTA ÇIKIŞ ÖZETİ (EPİKRİZ)
=========================================
1. HASTA DEMOGRAFİK BİLGİLERİ:
2. BAŞVURU ŞİKAYETİ VE ANAMNEZ (HİKAYE):
3. SİSTEM SORGULAMASI VE FİZİK MUAYENE:
4. LABORATUVAR VE GÖRÜNTÜLEME BULGULARI:
5. KLİNİK SEYİR VE KESİN TANI:
6. TABURCULUK TEDAVİSİ VE POLİKLİNİK ÖNERİLERİ:
=========================================
<|im_end|>
<|im_start|>user
Hocam, asistan notları şunlar: "{asistan_notu}"
Lütfen bu notları alıp yukarıdaki şablona göre muazzam detaylı ve resmi bir epikriz yazar mısınız?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("⏳ Kıdemli Doktor raporu dikte ediyor, lütfen bekleyin...")

with torch.no_grad():
    cikti = model.generate(
        **inputs, 
        max_new_tokens=1000,       # Rapor uzun olsun diye sınırı 400 kelimeye çıkardık
        temperature=0.3,          # Çok hafif bir esneklik verdik ki cümleleri güzel bağlasın
        top_p=0.9,
        repetition_penalty=1.1    # Tekrar etmesini önlemek için
    )

sonuc = tokenizer.decode(cikti[0], skip_special_tokens=True)
temiz_rapor = sonuc.split("assistant\n")[-1]

print("\n🏥 KIDEMLİ UZMAN DOKTORUN DETAYLI RAPORU:\n")
print(temiz_rapor)

⏳ Kıdemli Doktor raporu dikte ediyor, lütfen bekleyin...

🏥 KIDEMLİ UZMAN DOKTORUN DETAYLI RAPORU:

T.C. SAĞLIK BAKANLIĞI
HASTA ÇIKIŞ ÖZETİ (EPİKRİZ)
1. HASTA DEMOGRAFİK BİLGİLERİ:
   - Yaş: 30 yaş
   - Cinsiyet: Kadın
   - Adres: [Adres bilgisi]
   - Telefon: [Telefon numarası]
   - E-posta: [E-posta adresi]

2. BAŞVURU ŞİKAYETİ VE ANAMNEZ (HİKAYE):
   - Tarafından başvurulan tarih: [Tarafından başvurulan tarih]
   - Başvuru sebebi: [Başvuru sebebi]
   - Anamnesesi: [Anamnesesi]
   
3. SİSTEM SORGULAMASI VE FİZİK MUAYENE:
   - SYSTEMIC LUPUS ERITEMATOZUS (SLE): ONUNDA DOLAYISA DEĞİŞKENLER:
     - ANA (ANTINÜKLEER ANTIKOR): POSITIF
     - ANTI-DSDNA: POSITIVELDI
   
4. LABORATUVAR VE GÖRÜNTÜLEME BULGULARI:
   - Laboratuvar bulguları: 
     - ANA (Antinükleer Antikor): POSITIF
     - Anti-DsDNA: POSITIVELDI
   
5. KLİNİK SEYİR VE KESİN TANI:
   - Hastada SLE düşünüldüğü için, sistematik olarak kullanılan tedaviler arasında:
     - Antinekrozyonlar
     - Antihistaminikler
     - Antikok

In [7]:
!pip install langchain langchain-community sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ------------------- -------------------- 262.1/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 2.6 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 2.7 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 4.2 MB/s eta 0:00:01
   -------------------- ------------------- 1.3/2.5 MB 3.2 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 3.9 MB/s eta 0:00:01
   ----------------------------------------

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# --- 1. MODELİ TEKRAR HAFIZAYA ALIYORUZ ---
model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
cache_klasoru = r"D:\huggingface_cache"

print("⏳ 1. Qwen 1.5B Modeli Hafızaya Alınıyor (Diskten okunuyor)...")
tokenizer = AutoTokenizer.from_pretrained(model_adi, cache_dir=cache_klasoru)
model = AutoModelForCausalLM.from_pretrained(
    model_adi,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=cache_klasoru
)
model.eval()

# --- 2. TIBBİ BİLGİ BANKASI (VEKTÖR DB) KURULUMU ---
print("⚙️ 2. Tıbbi Bilgi Bankası (Vektör Veri Tabanı) Oluşturuluyor...")
tibbi_veriler = [
    Document(page_content="Hastalık: Sistemik Lupus Eritematozus (SLE). Belirtiler: Yüzde kelebek döküntüsü, poliartralji (eklem ağrısı), şiddetli yorgunluk, fotosensitivite. Laboratuvar: ANA pozitif, Anti-dsDNA pozitif. Tedavi: Hidroksiklorokin (Antimalaryal), düşük doz kortikosteroidler, güneşten korunma. Poliklinik Takibi: 1 ay sonra romatoloji kontrolü."),
    Document(page_content="Hastalık: Dermatomiyozit. Belirtiler: Göz etrafında heliotrop döküntü, kas güçsüzlüğü, Gottron papülleri. Laboratuvar: CK yüksekliği, Anti-Jo1 pozitif. Tedavi: Yüksek doz sistemik kortikosteroidler, immünsüpresanlar."),
    Document(page_content="Hastalık: Wilson Hastalığı. Belirtiler: Karaciğer enzim yüksekliği, nörolojik bozukluklar, Kayser-Fleischer halkası. Laboratuvar: Serum seruloplazmin düşüklüğü, 24 saatlik idrarda bakır atılımı yüksekliği. Tedavi: D-Penisilamin, Çinko tedavisi, bakırdan kısıtlı diyet.")
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vektor_db = FAISS.from_documents(tibbi_veriler, embeddings)
arama_motoru = vektor_db.as_retriever(search_kwargs={"k": 1})

# --- 3. LANGCHAIN & RAG MİMARİSİ ---
print("🔗 3. LangChain Ajan Zinciri Kuruluyor...")
pipe = pipeline(
    "text-generation",
    model=model,        
    tokenizer=tokenizer, 
    max_new_tokens=1500,
    temperature=0.01,   # Uydurmayı KESİNLİKLE yasaklıyoruz!
    repetition_penalty=1.1,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

prompt_sablonu = """<|im_start|>system
Sen uzman bir Romatoloji Doktorusun. KESİNLİKLE aşağıda sana verilen "TIBBİ BİLGİ KAYNAĞI" dışından hiçbir bilgi (ilaç, tedavi, kelime) uydurmayacaksın.
Sadece kaynakta yazanları kullanarak aşağıdaki şablonu doldur.

TIBBİ BİLGİ KAYNAĞI:
{kaynak_bilgi}

ŞABLON:
=========================================
T.C. SAĞLIK BAKANLIĞI
HASTA ÇIKIŞ ÖZETİ (EPİKRİZ)
=========================================
1. HASTA DEMOGRAFİK BİLGİLERİ: [Hastanın yaşı ve cinsiyeti]
2. BAŞVURU ŞİKAYETİ: [Şikayetlerin listesi]
3. LABORATUVAR BULGULARI: [Tahlil sonuçları]
4. KESİN TANI: [Hastalığın adı]
5. TABURCULUK TEDAVİSİ VE ÖNERİLER: [Kaynak bilgide yazan tedaviler ve poliklinik önerisi]
=========================================
<|im_end|>
<|im_start|>user
Hasta: {hasta_notu}<|im_end|>
<|im_start|>assistant
"""

prompt = PromptTemplate.from_template(prompt_sablonu)
ajan_zinciri = prompt | llm  

# --- 4. SİSTEMİ ÇALIŞTIRMA (TEST) ---
print("🚀 4. Ajan Çalıştırılıyor ve Rapor Hazırlanıyor...\n")

asistan_notu = "30 yaşında kadın hasta. Yüzünde kelebek döküntüsü, eklem ağrısı ve halsizliği var. Kanında ANA ve Anti-dsDNA pozitif geldi."

bulunan_belgeler = arama_motoru.invoke(asistan_notu)
getirilen_tibbi_bilgi = bulunan_belgeler[0].page_content

sonuc = ajan_zinciri.invoke({"kaynak_bilgi": getirilen_tibbi_bilgi, "hasta_notu": asistan_notu})

print("🏥 RAG MİMARİSİ İLE ÜRETİLEN KUSURSUZ RAPOR:\n")
print(sonuc.strip())

⏳ 1. Qwen 1.5B Modeli Hafızaya Alınıyor (Diskten okunuyor)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

⚙️ 2. Tıbbi Bilgi Bankası (Vektör Veri Tabanı) Oluşturuluyor...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🔗 3. LangChain Ajan Zinciri Kuruluyor...


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 4. Ajan Çalıştırılıyor ve Rapor Hazırlanıyor...

🏥 RAG MİMARİSİ İLE ÜRETİLEN KUSURSUZ RAPOR:

T.C. SAĞLIK BAKANLIĞI
HASTA ÇIKIŞ ÖZETİ (EPİKRİZ)
1. HASTA DEMOGRAFİK BİLGİLERİ: 30 yaşındaki kadın
2. BAŞVURU ŞİKAYETİ: Kelebek döküntüsü, eklem ağrısı, halsizlik, ANA pozitif, Anti-dsDNA pozitif
3. LABORATUVAR BULGULARI: ANA pozitif, Anti-dsDNA pozitif
4. KESİN TANI: Sistemik Lupus Eritematozus (SLE)
5. TABURCULUK TEDAVİSİ VE ÖNERİLER: Hidroksiklorokin (Antimalaryal), düşük doz kortikosteroidler, güneşten korunma
